## Full training loop

In [1]:
import torch

#### Neural network module 

In [2]:
class NeuralNetwork(torch.nn.Module): #create from module to inherit pytorch functionality
    def __init__(self, num_inputs, num_outputs): #constructor, takes in number of inputs and outputs for the network
        super().__init__() 

        self.layers = torch.nn.Sequential ( #allows us to stack layers in order with data flow easily.
            #layer 1 
            torch.nn.Linear(num_inputs, 30), #30 neurons each with weights and bias, Liner: y = weighted sum of inputs + bias
            torch.nn.ReLU(), #activation functions, ReLU: y = max(0,x) if neg ignore

            #layer 2
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),

            #output layer
            torch.nn.Linear(20, num_outputs) #produces desired outputs for prediction 
        )
    
    #take input x and push through layers to get output
    def forward(self,x): #how the data flows 
        logits = self.layers(x) #pass data through layers
        return logits
    #remember backpropagation is done by pytorch automatically 

#### Dataloader for the Data

In [3]:
X_train = torch.tensor([ # training data
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5]
])

#target labels
#output nodes = 2 (0 or 1) 
y_train = torch.tensor([0,0,0,1,1])


#test data 
X_test = torch.tensor([
    [-0.8,2.8],
    [2.6, -1.6]
])

y_test = torch.tensor([0,1])

In [4]:
from torch.utils.data import Dataset

class ToyDataset(Dataset):
    def __init__(self, X, y):
        #saves X_train and y_train as attributes of the class
        self.features = X
        self.labels = y
    
    def __getitem__(self, index):
        #when called it returns the feature and corresponding label of the index passed to it
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y
    
    def __len__(self):
        #returns the number of samples in the dataset
        return self.labels.shape[0]

train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)


In [5]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    dataset=train_ds, #uses the train_ds dataset and functions as an iterator to loop through the dataset in batches
    batch_size=2, #feeding 2 examples at a time to the model
    shuffle=True, 
    num_workers=0, # number of subprocesses to use for data loading. 0 means the data will be loaded in the main process
    drop_last=True #if the number of samples in the dataset is not divisible by the batch size, the last batch will be dropped
)

test_loader = DataLoader(
    dataset=test_ds,
    batch_size=2,
    shuffle=False,
    num_workers=0
)


#### Training


In [6]:
import torch.nn.functional as F 

torch.manual_seed(123)

#2 input nodes(2 features, 2 values per row) and 2 output nodes (0 or 1)
model = NeuralNetwork(num_inputs = 2, num_outputs = 2)

#initializes the gradient descent optimizer
optimizer = torch.optim.SGD(
    model.parameters(),
    lr = 0.5
)

num_epochs = 3 

#loop through data for the number of epochs specified 
for epoch in range(num_epochs):
    model.train() #turn on training mode 
    for batch_idx, (features, labels) in enumerate(train_loader): #train_loader based of Dataloader hands us one batch at a time
        
        #forward pass: making a prediction 
        logits = model(features)

        #loss calculation: how far off the prediction was 
        loss = F.cross_entropy(logits, labels)

        #backward pass: calculate gradients using chain rule to see how much each param added to the error
        loss.backward()

        #update weights: based off the gradients 
        optimizer.step()

        #reset gradients for next loop 
        optimizer.zero_grad()

        print(f"Epoch: {epoch+1: 03d}/{num_epochs:03d}"
                f" | Batch: {batch_idx+1:03d}/{len(train_loader):03d}"
                f" | Train Loss: {loss:.2f}")

Epoch:  01/003 | Batch: 001/002 | Train Loss: 0.75
Epoch:  01/003 | Batch: 002/002 | Train Loss: 0.65
Epoch:  02/003 | Batch: 001/002 | Train Loss: 0.44
Epoch:  02/003 | Batch: 002/002 | Train Loss: 0.13
Epoch:  03/003 | Batch: 001/002 | Train Loss: 0.03
Epoch:  03/003 | Batch: 002/002 | Train Loss: 0.00


After training model we can use it to make predictions!

In [8]:
model.eval()
with torch.no_grad():
    outputs = model(X_train)
print(outputs)

#convert logits to probabilities using softmax
torch.set_printoptions(sci_mode=False) 
probas = torch.softmax(outputs, dim=1)
print(probas)
#99% to class 0 and 0.000% to class 1

#convert probabilites to predicted class labels (show highest probability)
predictions = torch.argmax(probas, dim = 1)
print (predictions)








tensor([[ 2.8569, -4.1618],
        [ 2.5382, -3.7548],
        [ 2.0944, -3.1820],
        [-1.4814,  1.4816],
        [-1.7176,  1.7342]])
tensor([[0.9991, 0.0009],
        [0.9982, 0.0018],
        [0.9949, 0.0051],
        [0.0491, 0.9509],
        [0.0307, 0.9693]])
tensor([0, 0, 0, 1, 1])


we can compared the computed labels to the training dataset 

In [ ]:
predictions == y_train

torch.sum(predictions == y_train)#how many pred correct

tensor(5)

#### Generalized prediction accuracy function 

In [ ]:
def compute_accuracy(model, dataloader):

    model.eval() 
    correct = 0.0 
    total_examples = 0 

    for idx, (features,labels) in enumerate(dataloader):

        with torch.no_grad():
            logits = model(features)#run the model on the features to get the raw output values (logits)

        predictions = torch.argmax(logits,dim=1) #convert logits to predicted class label
        compare = labels == predictions #if label matches prediction True
        correct += torch.sum(compare) #add number of correct preidctions 
        total_examples += len(compare) #add number of examples gone through

    return (correct / total_examples).item()

print (compute_accuracy(model, train_loader))

print(compute_accuracy(model, test_loader))

1.0
1.0
